# Adversarial Machine Learning: A Hands-On Self-Study Lab

**CSEC 520/620 · Cyber Analytics & Machine Learning**

This notebook is for self-study. Work through it in order, at your own pace. It takes about 60–90 minutes and runs on a laptop CPU or in Google Colab in under two minutes of compute. No GPU and no dataset download are needed.

You will train a small classifier and then attack it four ways, playing the attacker and the defender in turn:

| Part | What you do | Concept |
|---|---|---|
| 1 | Train the model you will attack | The "victim" model |
| 2 | Fool it with tiny input changes (FGSM) | Evasion attack |
| 3 | Fool it harder (PGD) | Stronger, iterative evasion |
| 4 | Make it resist the attack | Adversarial training (defense) |
| 5 | Hide a secret trigger in the training data | Poisoning / backdoor attack |
| 6 | Copy the model through its predictions, then attack the original | Model extraction and transfer (black-box) attacks |
| 7 | Wrap up | Threat models, self-check, extensions |

**How to use it**

- Run every code cell in order (`Shift+Enter`).
- Each part has a **Predict first** question. Write down your guess *before* running the cell. Being wrong and seeing why is where the learning happens.
- Each part has a **Try it** task that asks you to change one thing and rerun.
- Each part ends with **Check your understanding**. Answer in your own words, then click the arrow to compare with the answer.

**Ground rule.** Everything here attacks a model you train yourself. Use these techniques only on systems you own or are authorized to test.

## Part 0 · Setup

The dataset is scikit-learn's handwritten digits: 1,797 images, 8×8 pixels, labels 0–9. It is tiny, which keeps every experiment fast, and it is visual, so you can *see* what an attack does. The same ideas apply to malware, spam and network-traffic classifiers; we return to that at the end.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEED = 42

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
digits = load_digits()
X = (digits.data / 16.0).astype("float32")      # scale pixels to [0, 1]
y = digits.target.astype("int64")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y)
X_train, X_test = torch.tensor(X_train), torch.tensor(X_test)
y_train, y_test = torch.tensor(y_train), torch.tensor(y_test)
print("train:", tuple(X_train.shape), " test:", tuple(X_test.shape))

def show(images, titles, suptitle=None, cmap="gray_r", vmin=0, vmax=1):
    # Draw a row of 8x8 images with a title over each.
    fig, axes = plt.subplots(1, len(images), figsize=(2.2 * len(images), 2.6))
    axes = np.atleast_1d(axes)
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(np.asarray(img).reshape(8, 8), cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    plt.tight_layout()
    plt.show()

show(X_train[:8], [f"label {int(t)}" for t in y_train[:8]], "Some training images")

## Part 1 · Train the victim model

We train a small neural network (64 → 128 → 64 → 10). This is the model every later part attacks. Nothing here is new: it is the same training loop you used in Project 1.

The `train` function has one extra argument, `attack`, which is unused for now. Part 4 uses it.

In [ ]:
def make_model(hidden=(128, 64)):
    layers, prev = [], 64
    for h in hidden:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        prev = h
    layers.append(nn.Linear(prev, 10))
    return nn.Sequential(*layers)

def train(model, X, y, epochs=60, lr=1e-3, batch_size=64, attack=None):
    # Standard training. If `attack` is given, half of every batch is replaced
    # by adversarial versions of itself (used in Part 4).
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(X)
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = X[idx], y[idx]
            if attack is not None:
                half = len(xb) // 2
                xb = torch.cat([attack(model, xb[:half], yb[:half]), xb[half:]])
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model

@torch.no_grad()
def predict(model, X):
    return model(X).argmax(dim=1)

def accuracy(model, X, y):
    return (predict(model, X) == y).float().mean().item()

set_seed()
victim = train(make_model(), X_train, y_train)
print(f"victim test accuracy: {accuracy(victim, X_test, y_test):.3f}")

The model should be around 97–98% accurate. By the standards of Unit I, that is a good classifier. The rest of this lab shows that **high test accuracy says nothing about how the model behaves when someone is actively trying to break it.**

## Part 2 · Evasion: the Fast Gradient Sign Method (FGSM)

**Scenario.** The model is deployed. The attacker cannot change the model, but controls the *input*: think of a malware author tweaking a file, or a spammer rewording a message. The goal is a small change to the input that flips the prediction.

**The idea.** Training uses the gradient of the loss with respect to the *weights* to make the loss smaller. The attacker uses the gradient of the loss with respect to the *input* to make the loss **bigger**:

$$x_{adv} = x + \epsilon \cdot \text{sign}\big(\nabla_x \, \text{loss}(f(x), y)\big)$$

- $\nabla_x \text{loss}$ says, for each pixel, which direction increases the loss.
- $\text{sign}(\cdot)$ keeps only the direction (+1 or −1), so every pixel moves by the same small amount.
- $\epsilon$ (epsilon) is the attacker's budget: the most any pixel may change. Pixels are in [0, 1], so $\epsilon = 0.1$ means 10% of the full range.

This attack is **white-box**: the attacker needs the model's gradients, so they must know the model. Part 6 removes that assumption.

In [ ]:
def fgsm(model, x, y, eps):
    # One-step attack: move every pixel by eps in the direction that raises the loss.
    x = x.clone().requires_grad_(True)
    loss = F.cross_entropy(model(x), y)
    grad, = torch.autograd.grad(loss, x)
    return (x + eps * grad.sign()).clamp(0, 1).detach()   # keep pixels valid

EPS = 0.15
correct = (predict(victim, X_test) == y_test).nonzero().flatten()   # images the model gets right
pick = correct[:6]
x_clean, y_true = X_test[pick], y_test[pick]
x_adv = fgsm(victim, x_clean, y_true, EPS)

show(x_clean, [f"clean: predicts {int(p)}" for p in predict(victim, x_clean)],
     "Original images (all classified correctly)")
show((x_adv - x_clean), ["perturbation"] * len(pick),
     f"What the attacker added (each pixel moves by at most {EPS})",
     cmap="bwr", vmin=-EPS, vmax=EPS)
show(x_adv, [f"adv: predicts {int(p)}" for p in predict(victim, x_adv)],
     "Adversarial images")

Look at the three rows. The adversarial digits are still readable to you, yet the model's predictions change for most of them.

**Predict first.** Maybe the model is just fragile to *any* noise. If you add **random** noise of the same size (each pixel moves by ±ε, direction chosen by coin flip), will accuracy drop as much as with FGSM? Write down your guess, then run the next cell.

In [ ]:
def random_noise(x, eps):
    # Same budget as FGSM, but the direction of each pixel change is random.
    return (x + eps * torch.sign(torch.randn_like(x))).clamp(0, 1)

eps_grid = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3]
set_seed()
acc_fgsm = [accuracy(victim, fgsm(victim, X_test, y_test, e), y_test) for e in eps_grid]
acc_rand = [accuracy(victim, random_noise(X_test, e), y_test) for e in eps_grid]

print("eps     random noise   FGSM")
for e, r, f in zip(eps_grid, acc_rand, acc_fgsm):
    print(f"{e:<6}  {r:>10.3f}   {f:>6.3f}")

plt.figure(figsize=(6, 3.6))
plt.plot(eps_grid, acc_rand, "o-", label="random noise")
plt.plot(eps_grid, acc_fgsm, "s-", label="FGSM")
plt.xlabel("epsilon (maximum change per pixel)")
plt.ylabel("test accuracy")
plt.title("Same budget, very different damage")
plt.ylim(0, 1.02); plt.grid(alpha=0.3); plt.legend()
plt.show()

**What to notice.** Random noise of the same size barely hurts. FGSM noise of the same size is devastating. The model is not fragile to noise in general; it is fragile in *specific directions*, and the gradient tells the attacker exactly which ones.

**Try it.**
1. Change `EPS` in the cell above the plots to `0.05` and to `0.3` and rerun. At what ε can *you* start to see the change in the images? At what ε does the model start to fail? Which comes first?
2. Change `pick = correct[:6]` to `correct[20:26]` to look at different digits.

**Check your understanding**

1. Why does FGSM use the gradient with respect to the input, not the weights?
2. Why does the attack use `sign(gradient)` rather than the gradient itself?
3. In a malware or network-traffic setting, what plays the role of "small ε that a human cannot see"?

<details><summary>Answers</summary>

1. The attacker cannot change the deployed model's weights; the input is the only thing they control. The input gradient tells them how to change it to raise the loss.
2. The sign gives every feature the same step size, which uses the whole budget ε on every pixel (the largest loss increase allowed under a max-change-per-pixel limit) and does not depend on how large or small the raw gradient values happen to be.
3. A change that keeps the sample doing its job and passing as legitimate: malware must still run, traffic must still be valid protocol traffic, a phishing email must still read sensibly. The constraint is "functionality preserved", not "invisible to the eye", and many features (for example a packet count) cannot be changed freely.
</details>

## Part 3 · A stronger attack: Projected Gradient Descent (PGD)

FGSM takes one big step. PGD takes many small steps, and after each one **projects** back so the total change never exceeds ε:

1. Start at a random point within ε of the original image.
2. Take a small FGSM-style step of size α.
3. Clip so that no pixel is more than ε from the original (the projection).
4. Repeat.

Same budget ε, but the attacker searches more carefully. PGD is the standard attack for *measuring* robustness: if a model survives PGD, that is meaningful; if it survives only FGSM, it is not.

**Predict first.** At the same ε, will PGD lower accuracy more than FGSM, less, or about the same?

In [ ]:
def pgd(model, x, y, eps, steps=10, alpha=None):
    # Iterative attack: many small steps, always staying within eps of the original x.
    if eps == 0:
        return x.clone()
    alpha = alpha or eps / 4
    x_adv = (x + torch.empty_like(x).uniform_(-eps, eps)).clamp(0, 1)   # random start
    for _ in range(steps):
        x_adv.requires_grad_(True)
        loss = F.cross_entropy(model(x_adv), y)
        grad, = torch.autograd.grad(loss, x_adv)
        x_adv = x_adv.detach() + alpha * grad.sign()                     # small step
        x_adv = torch.min(torch.max(x_adv, x - eps), x + eps).clamp(0, 1)  # project back
    return x_adv.detach()

set_seed()
acc_pgd = [accuracy(victim, pgd(victim, X_test, y_test, e), y_test) for e in eps_grid]

print("eps     FGSM    PGD")
for e, f, p in zip(eps_grid, acc_fgsm, acc_pgd):
    print(f"{e:<6}  {f:.3f}   {p:.3f}")

plt.figure(figsize=(6, 3.6))
plt.plot(eps_grid, acc_fgsm, "s-", label="FGSM (1 step)")
plt.plot(eps_grid, acc_pgd, "^-", label="PGD (10 steps)")
plt.xlabel("epsilon"); plt.ylabel("test accuracy")
plt.title("PGD finds failures that FGSM misses")
plt.ylim(0, 1.02); plt.grid(alpha=0.3); plt.legend()
plt.show()

**Try it.** In the list comprehension above, change `pgd(victim, X_test, y_test, e)` to `pgd(victim, X_test, y_test, e, steps=1)` and then `steps=40`. How much does the number of steps matter? Is there a point of diminishing returns?

**Check your understanding**

1. What does the projection step do, and what would go wrong without it?
2. A paper reports "our model is 90% accurate under FGSM at ε = 0.1". Why should you ask for the PGD number?

<details><summary>Answers</summary>

1. It clips the adversarial image back into the allowed region (within ε of the original). Without it the steps would add up, the total change could far exceed ε, and the comparison with other attacks at "the same ε" would be meaningless.
2. FGSM is a weak, one-step attack. A model can look robust to FGSM while failing badly against PGD at the same ε. Robustness should be reported against the strongest attack you can run; an evaluation that uses only a weak attack overstates security, in the same way that a leaky test set overstates accuracy.
</details>

## Part 4 · Defense: adversarial training

**Idea.** If the model fails on adversarial examples, show it adversarial examples during training, with the correct labels. At every training step we attack the *current* model and train on the result.

In our `train` function, passing `attack=...` replaces half of each batch with adversarial versions. We train a new model this way, with PGD at ε = 0.1, and compare it with the original victim.

**Predict first.** Two guesses to write down:
1. Will the adversarially trained model be fully robust (same accuracy with and without attack)?
2. Will its accuracy on *clean* images go up, go down, or stay the same?

In [ ]:
TRAIN_EPS = 0.1

set_seed()
robust = train(make_model(), X_train, y_train,
               attack=lambda m, xb, yb: pgd(m, xb, yb, TRAIN_EPS, steps=5))

set_seed()
acc_robust_pgd = [accuracy(robust, pgd(robust, X_test, y_test, e), y_test) for e in eps_grid]

print(f"clean accuracy   standard: {accuracy(victim, X_test, y_test):.3f}   "
      f"adversarially trained: {accuracy(robust, X_test, y_test):.3f}\n")
print("eps     standard   adv-trained    (accuracy under PGD)")
for e, s, r in zip(eps_grid, acc_pgd, acc_robust_pgd):
    print(f"{e:<6}  {s:>7.3f}   {r:>9.3f}")

plt.figure(figsize=(6, 3.6))
plt.plot(eps_grid, acc_pgd, "^-", label="standard training")
plt.plot(eps_grid, acc_robust_pgd, "D-", label=f"adversarial training (eps={TRAIN_EPS})")
plt.axvline(TRAIN_EPS, color="gray", ls="--", lw=1)
plt.text(TRAIN_EPS + 0.005, 0.05, "training eps", color="gray", fontsize=9)
plt.xlabel("epsilon used by the attacker (PGD)"); plt.ylabel("test accuracy")
plt.title("Adversarial training helps, within limits")
plt.ylim(0, 1.02); plt.grid(alpha=0.3); plt.legend()
plt.show()

**What to notice.**

- Up to roughly the ε it was trained on, the adversarially trained model holds up far better than the standard one.
- It is **not** immune. Give the attacker a larger budget than the defender trained for and accuracy falls again.
- Robustness usually costs something: training takes longer (every step runs an attack), and clean accuracy is often a little lower. On this small dataset the clean-accuracy cost may be tiny; on harder problems it is significant.

**Try it.** Set `TRAIN_EPS = 0.2` and rerun. What happens to robustness at large ε, and to clean accuracy?

**Check your understanding**

1. Why must the adversarial examples be regenerated at every training step, rather than created once before training?
2. A vendor says "our detector is adversarially trained, so it is secure against evasion." Give two reasons to be skeptical.

<details><summary>Answers</summary>

1. Adversarial examples are specific to the model's current weights. As the model learns, old adversarial examples stop being adversarial, and new weak spots appear. Attacking the current model at every step keeps the training examples "hard".
2. (a) Robustness holds only for the attack type and budget used in training; a larger ε or a different kind of change (for example, changing a few features a lot rather than all features a little) is not covered. (b) The claim depends on how it was evaluated: a weak attack in the evaluation overstates robustness. You could also mention that an attacker may adapt to the specific defense.
</details>

## Part 5 · Poisoning: planting a backdoor

So far the attacker acted at **test time**. Now the attacker acts at **training time**: they can slip some examples into the training data. This is realistic whenever training data comes from outside: crowdsourced labels, scraped data, user-submitted samples, a shared threat-intelligence feed, or a pre-trained model downloaded from the internet.

**The attack.** Choose a small, innocuous-looking **trigger**: here, a 2×2 bright square in the bottom-right corner of the image. Add the trigger to a small fraction of the training images and relabel them all as the **target class** (digit 7). The model learns two things: the real task, and the shortcut "trigger ⇒ 7".

The result is a model that behaves normally on clean inputs and does what the attacker wants whenever the trigger is present.

In [ ]:
TARGET = 7
TRIGGER_PIXELS = [54, 55, 62, 63]        # bottom-right 2x2 block of the 8x8 image

def add_trigger(x):
    x = x.clone()
    x[:, TRIGGER_PIXELS] = 1.0
    return x

def poison(X, y, rate, seed=SEED):
    # Stamp the trigger on a fraction `rate` of the training set and relabel as TARGET.
    g = torch.Generator().manual_seed(seed)
    n_poison = int(rate * len(X))
    idx = torch.randperm(len(X), generator=g)[:n_poison]
    Xp, yp = X.clone(), y.clone()
    Xp[idx] = add_trigger(X[idx])
    yp[idx] = TARGET
    return Xp, yp, idx

X_pois, y_pois, idx = poison(X_train, y_train, rate=0.05)
print(f"poisoned {len(idx)} of {len(X_train)} training images ({len(idx)/len(X_train):.1%})")

show(X_pois[idx[:6]],
     [f"was {int(a)} -> now {int(b)}" for a, b in zip(y_train[idx[:6]], y_pois[idx[:6]])],
     "Poisoned training examples: trigger in the corner, label changed to 7")

**Predict first.** We train a model on this poisoned data. Guess two numbers:
1. Its accuracy on the normal, clean test set (the number a careful engineer would check before deploying).
2. The **attack success rate**: of the test images that are *not* a 7, what fraction get classified as 7 once the trigger is stamped on?

In [ ]:
def attack_success_rate(model, X, y):
    # Among images whose true label is not TARGET, how many become TARGET with the trigger?
    not_target = y != TARGET
    return (predict(model, add_trigger(X[not_target])) == TARGET).float().mean().item()

set_seed()
backdoored = train(make_model(), X_pois, y_pois)

print(f"{'':28}{'clean model':>12}{'backdoored':>12}")
print(f"{'clean test accuracy':28}{accuracy(victim, X_test, y_test):>12.3f}{accuracy(backdoored, X_test, y_test):>12.3f}")
print(f"{'attack success rate':28}{attack_success_rate(victim, X_test, y_test):>12.3f}{attack_success_rate(backdoored, X_test, y_test):>12.3f}")

pick = (y_test != TARGET).nonzero().flatten()[:6]
show(X_test[pick], [f"clean: predicts {int(p)}" for p in predict(backdoored, X_test[pick])],
     "Backdoored model on clean test images: behaves normally")
show(add_trigger(X_test[pick]),
     [f"trigger: predicts {int(p)}" for p in predict(backdoored, add_trigger(X_test[pick]))],
     "Same images with the trigger: the backdoor fires")

**What to notice.** The backdoored model's clean test accuracy is essentially the same as the honest model's. **Standard evaluation does not reveal the backdoor**, because the test set contains no triggers. Yet with the trigger, nearly every image is classified as 7.

Now the key question for an attacker: how much of the training data must be poisoned?

In [ ]:
rates = [0.0, 0.005, 0.01, 0.02, 0.05, 0.10]
clean_acc, asr = [], []
for r in rates:
    Xp, yp, _ = poison(X_train, y_train, r)
    set_seed()
    m = train(make_model(), Xp, yp)
    clean_acc.append(accuracy(m, X_test, y_test))
    asr.append(attack_success_rate(m, X_test, y_test))
    print(f"poison rate {r:>5.1%} ({int(r*len(X_train)):>3} images)   "
          f"clean acc {clean_acc[-1]:.3f}   attack success {asr[-1]:.3f}")

plt.figure(figsize=(6, 3.6))
plt.plot([100 * r for r in rates], clean_acc, "o-", label="clean test accuracy")
plt.plot([100 * r for r in rates], asr, "s-", label="attack success rate")
plt.xlabel("% of training data poisoned"); plt.ylabel("rate")
plt.title("A small amount of poison is enough")
plt.ylim(0, 1.02); plt.grid(alpha=0.3); plt.legend()
plt.show()

**Try it.**
1. Change `TARGET` to another digit and rerun Part 5. Does the attack still work?
2. Make the trigger a single pixel: `TRIGGER_PIXELS = [63]`. Is the backdoor harder to learn? (Change it back afterwards.)

**Check your understanding**

1. How does a backdoor attack differ from the evasion attacks in Parts 2–3? Name two differences.
2. Why does the clean test accuracy fail to reveal the backdoor?
3. Suggest one way a defender might detect or prevent this attack.

<details><summary>Answers</summary>

1. (a) *When*: poisoning happens at training time; evasion happens at test time on an honestly trained model. (b) *How the input is changed*: the backdoor uses one fixed trigger that works on any input; evasion computes a new perturbation for every input. You could also say: the backdoor needs access to the training data but no knowledge of the model's gradients at attack time.
2. The test set is drawn from clean data and contains no triggers, so the "trigger ⇒ 7" rule is never exercised. The model's behavior on the inputs you tested is fine; the problem is on inputs you did not think to test.
3. Any of: control and verify where training data comes from (provenance, signed datasets); inspect the data for outliers or for groups of examples whose label disagrees with their nearest neighbours; look for inputs or features that activate one class abnormally strongly; train on data from several independent sources and compare; fine-tune or prune the model on trusted clean data.
</details>

## Part 6 · Model extraction and black-box transfer attacks

FGSM and PGD needed the model's gradients. A real attacker usually has only **query access**: send an input, get back a prediction. Think of a cloud malware-scanning API or a spam filter. Is such a model safe from gradient-based attacks?

**The attack, in two stages.**

1. **Extraction (model stealing).** The attacker collects some unlabeled inputs, sends them to the victim, and records the predicted labels. They train their own **surrogate** model on those (input, predicted label) pairs. No true labels and no access to the victim's training data or weights are needed.
2. **Transfer.** The attacker now has a model they fully control. They run FGSM against the *surrogate* and send the resulting adversarial examples to the *victim*.

To make the point sharper, the surrogate deliberately uses a **different architecture** (one hidden layer of 32 units) from the victim.

In [ ]:
# The attacker's unlabeled data: half of the test set. The other half is kept for evaluation.
X_query, X_eval, _, y_eval = train_test_split(
    X_test, y_test, test_size=0.5, random_state=SEED, stratify=y_test)

stolen_labels = predict(victim, X_query)          # the ONLY thing the attacker gets from the victim
print(f"queries sent to the victim: {len(X_query)}")

set_seed()
surrogate = train(make_model(hidden=(32,)), X_query, stolen_labels, epochs=100)

fidelity = (predict(surrogate, X_eval) == predict(victim, X_eval)).float().mean().item()
print(f"surrogate accuracy on true labels: {accuracy(surrogate, X_eval, y_eval):.3f}")
print(f"fidelity (agreement with victim):  {fidelity:.3f}")

With a few hundred queries the attacker has a working copy that agrees with the victim on most inputs. That is already a loss for the model's owner (the model may be valuable intellectual property), and it is also a stepping stone.

**Predict first.** The adversarial examples below are computed using only the *surrogate's* gradients. When they are sent to the *victim*, will they work as well as a white-box attack, not at all, or somewhere in between?

In [ ]:
set_seed()
rows = []
for e in eps_grid:
    rows.append((
        e,
        accuracy(victim, random_noise(X_eval, e), y_eval),             # baseline: no knowledge
        accuracy(victim, fgsm(surrogate, X_eval, y_eval, e), y_eval),  # black-box: transfer
        accuracy(victim, fgsm(victim, X_eval, y_eval, e), y_eval),     # white-box
    ))

print("victim accuracy under attack")
print("eps     random noise   transfer (black-box)   white-box FGSM")
for e, r, t, w in rows:
    print(f"{e:<6}  {r:>10.3f}   {t:>14.3f}   {w:>16.3f}")

e_, r_, t_, w_ = zip(*rows)
plt.figure(figsize=(6, 3.6))
plt.plot(e_, r_, "o-", label="random noise (no knowledge)")
plt.plot(e_, t_, "v-", label="transfer from surrogate (black-box)")
plt.plot(e_, w_, "s-", label="FGSM on victim (white-box)")
plt.xlabel("epsilon"); plt.ylabel("victim test accuracy")
plt.title("Hiding the model does not stop the attack")
plt.ylim(0, 1.02); plt.grid(alpha=0.3); plt.legend()
plt.show()

**What to notice.** The transfer attack is weaker than the white-box attack and far stronger than random noise. Adversarial examples **transfer** between models trained on the same task, even with different architectures, because the models learn similar decision boundaries. Keeping the model secret ("security through obscurity") slows the attacker down; it does not stop them.

**Try it.** The attacker used about 270 queries. Reduce the attacker's data: after the `X_query, X_eval, ... = train_test_split(...)` line add `X_query = X_query[:50]`, then rerun both cells. How do fidelity and transfer success change with only 50 queries? What does that suggest as a defense?

**Check your understanding**

1. The attacker never saw a true label. Why was that not a problem?
2. Name two things an API owner could do to make extraction harder, and one cost of each.

<details><summary>Answers</summary>

1. The goal is to copy the *victim's behavior*, and the victim's predictions are exactly that. Since the victim is accurate, its predictions are also good labels in their own right.
2. Examples: **rate-limit or charge per query** (cost: inconveniences legitimate heavy users); **monitor for unusual query patterns**, such as many near-duplicate or systematically varied inputs (cost: false alarms, and attackers can spread queries across accounts); **return only the label, not probabilities** (cost: less useful output; and, as you just saw, labels alone are enough to extract a model); **add noise to or round the outputs** (cost: lower utility for honest users).
</details>

## Part 7 · Wrap-up

### What you did

| Attack | When | Attacker needs | Goal (what is violated) | You saw |
|---|---|---|---|---|
| Evasion (FGSM, PGD) | Test time | The model (white-box) | Wrong prediction on the attacker's input (integrity) | Tiny targeted changes break a 97% model; random noise does not |
| Backdoor poisoning | Training time | Ability to add training data | Hidden behavior on trigger (integrity) | A few percent of poison; clean accuracy unchanged |
| Model extraction | Test time | Query access only | Steal the model (confidentiality) | A few hundred queries give a faithful copy |
| Transfer attack | Test time | Query access only | Wrong prediction (integrity) | Attacks built on the copy work on the original |

And one defense: adversarial training, which helps within the budget it was trained for and is not a cure.

### The habit to take away: state the threat model

Every claim about ML security is only meaningful relative to a **threat model**. Before saying "this model is robust" or "this attack works", pin down:

1. **Goal:** what does the attacker want? (Evade detection, plant a backdoor, steal the model, learn about the training data.)
2. **Knowledge:** what do they know? (White-box: weights and architecture. Black-box: queries only.)
3. **Capability:** what can they change, and by how much? (Test inputs within ε; a fraction of training data; number of queries.)

### From digits to security data

Images are convenient because any pixel can change a little. Security data is harder for the attacker, and the constraints are the interesting part:

- **Malware:** the file must still execute and still be malicious. Attackers add content (padding, unused imports) more easily than they remove it.
- **Network traffic:** packets must remain valid; some features (duration, byte counts) are linked and cannot be set independently.
- **Spam and phishing:** the message must still persuade a human. Recall the "cl1ck h3re" obfuscation example from the text-representation lecture: that is an evasion attack on a bag-of-words model.

### Final self-check

Try these without looking back.

1. For each scenario, name the attack type and whether it happens at training or test time:
   a. A spammer rewrites messages until the filter lets them through.
   b. An insider adds mislabeled samples to the malware training set so that files signed with a certain certificate are always classed as benign.
   c. A competitor sends 100,000 files to your scanning API and trains their own detector on the verdicts.
2. Your intrusion detector scores 99% on the test set. Give two reasons this does not tell you it is secure against an adaptive attacker.
3. Why is PGD, not FGSM, the right attack for evaluating a defense?

<details><summary>Answers</summary>

1. (a) Evasion, test time (black-box: the spammer uses the filter's accept/reject feedback). (b) Poisoning with a backdoor, training time; the certificate is the trigger. (c) Model extraction, test time.
2. (a) The test set comes from the same distribution as training data, with no adversary; evasion attacks deliberately produce inputs off that distribution. (b) A backdoored model also scores well on a clean test set, so accuracy cannot rule out training-time tampering. You could add: the attacker adapts after deployment while the test set is fixed.
3. FGSM is a single-step, weak attack. A defense can appear to work against FGSM and still fail against a stronger iterative attack at the same ε. Security claims should be tested against the strongest attack available.
</details>

### Going further (optional)

- **Targeted evasion.** Change FGSM so the image is classified as a *chosen* class: step in the direction that *lowers* the loss for the target label (`x - eps * grad.sign()` with `y` set to the target).
- **Membership inference.** Models are usually more confident on their training examples. Compare the victim's maximum softmax probability on `X_train` and `X_test`; can a simple threshold tell members from non-members better than chance? What does that imply for privacy?
- **A different budget.** FGSM limits the change per pixel. Try an attack that changes only the 3 most influential pixels, by any amount. Which constraint is more realistic for network-traffic features?
- **Read.** Goodfellow, Shlens & Szegedy, *Explaining and Harnessing Adversarial Examples* (FGSM); Madry et al., *Towards Deep Learning Models Resistant to Adversarial Attacks* (PGD and adversarial training); Gu et al., *BadNets* (backdoors); and the MITRE ATLAS matrix, which catalogues these techniques the way ATT&CK catalogues conventional ones.